# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA
* Model: GPT-2
* Evaluation approach: evaluate method with Hugging Face Trainer
* Fine-tuning dataset: sms dataset from Hugging Face's datasets library

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
# Load the sms_spam dataset
# See: https://huggingface.co/datasets/sms_spam

from datasets import load_dataset

# The sms_spam dataset only has a train split, so we use the train_test_split method to split it into train and test
dataset = load_dataset("sms_spam", split="train").train_test_split(
    test_size=0.2, shuffle=True, seed=23
)

splits = ["train", "test"]

# View the dataset characteristics
dataset["train"]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

Dataset({
    features: ['sms', 'label'],
    num_rows: 4459
})

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Let's use a lambda function to tokenize all the examples
tokenized_dataset = {}
for split in splits:
    tokenized_dataset[split] = dataset[split].map(
        lambda x: tokenizer(x["sms"], truncation=True), batched=True
    )

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2", 
    num_labels = 2,
    id2label={0: "not spam", 1: "spam"},
    label2id={"not spam": 0, "spam": 1},
)

model.config.pad_token_id = tokenizer.pad_token_id
    
# Unfreeze all the model parameters.
# Hint: Check the documentation at https://huggingface.co/transformers/v4.2.2/training.html
for param in model.parameters():
    param.requires_grad = True
    
config = LoraConfig(
    r=8, 
    lora_alpha=32, 
    target_modules=["c_attn"], # Correct for GPT-2
    lora_dropout=0.1, 
    task_type="SEQ_CLS"
)

lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 297,984 || all params: 124,737,792 || trainable%: 0.23888830740245906


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [3]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

# The HuggingFace Trainer class handles the training and eval loop for PyTorch for us.
# Read more about it here https://huggingface.co/docs/transformers/main_classes/trainer
trainer = Trainer(
    model=lora_model,
    args=TrainingArguments(
        output_dir="./tmp/model-final",
        # Set the learning rate
        learning_rate=2e-5,
        # Set the per device train batch size and eval batch size
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        # Evaluate and save the model after each epoch
        evaluation_strategy="epoch",
        save_strategy="epoch",
        # Set the learning rate
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.319552,0.872646
2,0.466400,0.168504,0.944395
3,0.466400,0.121582,0.973094
4,0.144500,0.100973,0.979372
5,0.144500,0.104042,0.979372


TrainOutput(global_step=1395, training_loss=0.24449971243472082, metrics={'train_runtime': 240.2157, 'train_samples_per_second': 92.812, 'train_steps_per_second': 5.807, 'total_flos': 695383346055168.0, 'train_loss': 0.24449971243472082, 'epoch': 5.0})

In [10]:
trainer.evaluate()

{'eval_loss': 0.10097304731607437,
 'eval_accuracy': 0.979372197309417,
 'eval_runtime': 5.3874,
 'eval_samples_per_second': 206.964,
 'eval_steps_per_second': 12.993,
 'epoch': 5.0}

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [13]:
# Saving the model
lora_model.save_pretrained("./tmp/gpt-lora-final")
tokenizer.save_pretrained("./tmp/gpt-lora-final")

('./tmp/gpt-lora-final/tokenizer_config.json',
 './tmp/gpt-lora-final/special_tokens_map.json',
 './tmp/gpt-lora-final/vocab.json',
 './tmp/gpt-lora-final/merges.txt',
 './tmp/gpt-lora-final/added_tokens.json',
 './tmp/gpt-lora-final/tokenizer.json')

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [14]:
import torch
from peft import AutoPeftModelForSequenceClassification
from transformers import AutoTokenizer

# load the model as a classifier
model_id = "./tmp/gpt-lora-final" 
lora_model = AutoPeftModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,    # match training setup
    device_map="auto"
)

# setup tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# run inference (prediction)
text = "1238201938091afjweiojfaoweijf10-2019238091289308"
inputs = tokenizer(text, return_tensors="pt").to(lora_model.device)

with torch.no_grad():
    logits = lora_model(**inputs).logits
    prediction = torch.argmax(logits, dim=-1).item()

# map back to labels
labels = {0: "not spam", 1: "spam"}
print(f"Prediction: {labels[prediction]}")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Prediction: spam


In [15]:
from transformers import Trainer

lora_model.config.pad_token_id = tokenizer.pad_token_id

# trainer for loaded model
final_trainer = Trainer(
    model=lora_model,
    args=TrainingArguments(
        output_dir="./tmp/fine-tuned-model-final",
        # Set the learning rate
        learning_rate=2e-5,
        # Set the per device train batch size and eval batch size
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        # Evaluate and save the model after each epoch
        evaluation_strategy="epoch",
        save_strategy="epoch",
        # Set the learning rate
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Final evaluation (Fine-Tuned Model):")
final_trainer.evaluate()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Final evaluation (Fine-Tuned Model):


{'eval_loss': 0.10097304731607437,
 'eval_accuracy': 0.979372197309417,
 'eval_runtime': 5.4152,
 'eval_samples_per_second': 205.903,
 'eval_steps_per_second': 12.927}